In [22]:
import faiss
import numpy as np
import polars as pl # 高速DataFrame库，用于数据处理与分析，速度更快，内存占用更低
from gensim.test.utils import common_texts
from gensim.models import Word2Vec

In [2]:
train=pl.read_parquet('./data/processData/train.parquet')
test=pl.read_parquet('./data/processData/test.parquet')

In [6]:
# pl.concat纵向合并，groupby按照session分组，agg聚合函数(不保留原维度，合并session)，pl.col选择列，alias重命名
# 为每个session合并aid组合成句子
sentences_df=pl.concat([train,test]).group_by('session').agg(pl.col('aid').alias('sentence'))

In [7]:
sentences_df

session,sentence
i32,list[i32]
2901729,"[802704, 802704]"
10393341,"[271233, 1526380, 1680161]"
8616946,"[1220318, 136121, 353807]"
1368751,"[459463, 157191, … 1030727]"
8687326,"[484932, 660775, … 633631]"
…,…
5322748,"[153997, 1145902, … 1145902]"
6057322,"[709399, 709399, … 166037]"
11233281,"[1189296, 1621255, … 1635950]"


In [8]:
sentences=sentences_df['sentence'].to_list()

In [10]:
sentences

[[802704, 802704],
 [271233, 1526380, 1680161],
 [1220318, 136121, 353807],
 [459463,
  157191,
  692445,
  1626591,
  331020,
  1559519,
  331020,
  1626591,
  364131,
  838586,
  838586,
  1145273,
  842086,
  842086,
  995701,
  995701,
  459166,
  995701,
  1829090,
  1829090,
  1030727],
 [484932, 660775, 1056119, 633631],
 [1580498, 174279, 1634198],
 [1490082,
  1755739,
  1405779,
  1026175,
  1026175,
  1490082,
  944474,
  790437,
  1796857,
  803138,
  155721,
  1182401,
  1772424,
  1772424,
  1293457,
  1772424,
  865607],
 [1498269, 1498269, 166037, 166037, 166037],
 [33635,
  1417307,
  1586818,
  752197,
  752197,
  752197,
  752197,
  1200389,
  579088,
  10851,
  10851,
  368900,
  1225851,
  1225851],
 [1640771, 725049, 1213113],
 [1662208, 1454065],
 [485847,
  1368794,
  1368794,
  1697138,
  688503,
  57804,
  407460,
  1760515,
  895243,
  609799,
  353442,
  353442,
  48483,
  48483,
  1770776,
  1179688,
  1606292,
  1606292,
  1346920,
  1606292,
  688503,
  1

In [15]:
import os
# 做embedding
model_path='./model/w2vec.model'
if os.path.exists(model_path):
    w2vec=Word2Vec.load(model_path)
else:
    # sentences训练语料，vector_size词向量维度，min_count此至少出现一次才会被训练，workers训练时使用的CPU线程数
    w2vec=Word2Vec(sentences=sentences,vector_size=32,min_count=1,workers=4)
    w2vec.save(model_path)

In [18]:
aids=w2vec.wv.index_to_key # 按照出现频率从高到低返回词的列表
aids

[1460571,
 485256,
 108125,
 29735,
 1733943,
 832192,
 184976,
 166037,
 554660,
 986164,
 231487,
 1502122,
 1603001,
 1236775,
 322370,
 332654,
 1196256,
 756588,
 959208,
 1083665,
 1022566,
 620545,
 95488,
 801774,
 247240,
 673407,
 1645990,
 1586171,
 508883,
 1116095,
 1294924,
 530377,
 811371,
 1604220,
 892871,
 152547,
 714524,
 102345,
 1531805,
 409620,
 670006,
 544144,
 1257293,
 1498443,
 1197632,
 199409,
 584027,
 819288,
 1796103,
 612920,
 1743151,
 496180,
 399315,
 636101,
 500609,
 33343,
 77440,
 1647563,
 1685214,
 632365,
 1581568,
 1462420,
 1043508,
 1182614,
 1264313,
 329725,
 861401,
 1658239,
 1497089,
 881286,
 326904,
 1111967,
 984459,
 137514,
 634452,
 794192,
 1006198,
 803928,
 1125638,
 11830,
 305158,
 1365988,
 721034,
 1406660,
 10964,
 331708,
 385065,
 1255910,
 1624436,
 1636724,
 1142000,
 190818,
 1419849,
 670066,
 159789,
 1338993,
 493104,
 884502,
 1629608,
 842590,
 450505,
 1610239,
 1052212,
 1551213,
 1383529,
 1116621,
 135997

In [37]:
aid2idx={aid:i for i,aid in enumerate(aids)}

# w2vec.wv[aid]可以检索到向量
vecs=[x for x in w2vec.wv.vectors] # 词的向量
d=w2vec.wv.vectors.shape[1]

In [38]:
vecs

[array([-1.2492757 ,  0.05667125,  4.3028946 ,  0.6818086 , -0.09911077,
        -0.5592608 ,  1.1714091 , -0.4163052 ,  0.8208092 , -0.65914387,
         2.366737  , -1.2084225 , -0.9860984 , -0.33512732, -0.58343416,
        -0.8540501 , -1.4526554 ,  1.0036968 , -2.4548113 ,  1.0161672 ,
        -0.07304483,  1.3423564 ,  1.4403344 ,  0.31924427,  0.7047517 ,
        -0.51282394, -0.7376928 ,  0.26792517, -0.94678926, -0.46386057,
        -0.43027028,  0.63884175], dtype=float32),
 array([-0.5322012 , -0.6342003 ,  2.1390097 , -0.9127722 ,  0.18830436,
        -0.4308944 ,  2.3791652 , -1.6030532 , -0.5250847 ,  0.34020138,
         0.25690073, -1.902897  , -0.885504  , -1.7102746 ,  0.16744632,
         0.45155707, -1.6804588 ,  2.5340712 , -0.9099356 ,  1.6073211 ,
         0.92997646,  2.3502076 ,  0.6436254 ,  0.49145883,  0.6578185 ,
        -2.0324843 , -1.4191215 , -0.28623655,  0.95618904, -0.57795817,
        -0.01725855,  0.6724278 ], dtype=float32),
 array([ 0.7866788 ,  

In [39]:
vecs=np.ascontiguousarray(vecs,dtype='float32')

In [40]:
type(vecs)

numpy.ndarray

In [42]:
import faiss
# 建立faiss索引,传入向量的维度
index=faiss.IndexFlatL2(d)
index.add(vecs)

ValueError: input not a numpy array